In [398]:
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, HDBSCAN
from sklearn.preprocessing import RobustScaler
from sklearn.metrics.pairwise import cosine_distances
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt

# load data

In [399]:
df = pd.read_csv('/Users/connorhall/datasets/inst414/module 3 assignment/Public.csv',
                 usecols=['SPECIES', 'AIRPORT', 'STATE', 'INCIDENT_DATE'])
df = df.dropna()

# remove unknown species
df = df[(~df['SPECIES'].str.contains('unknown|perching birds', case=False)) &
        (~df['AIRPORT'].str.contains('unknown', case=False))]
df

,INCIDENT_DATE,AIRPORT,STATE,SPECIES
8,1992-05-03,NORMAN Y. MINETA SAN JOSE INTL ARPT,CA,American robin
11,1995-04-14,ALEXANDRIA INTL,LA,Blackbirds
13,1994-09-01,SYRACUSE HANCOCK INTL,NY,Gulls
14,1990-09-17,OAKLAND COUNTY INTL,MI,Mourning dove
15,1990-07-13,LOS ANGELES INTL,CA,American kestrel
...,...,...,...,...
339266,2025-10-02,EVANSVILLE REGIONAL,IN,Chimney swift
339267,2025-10-01,SEATTLE-TACOMA INTL,WA,Cedar waxwing
339268,2025-10-02,CHICAGO O'HARE INTL ARPT,IL,Yellow-rumped warbler
339269,2025-10-03,ROANOKE REGNL ARPT/WOODRUM FIELD,VA,European starling


### filter data

In [400]:
# preview data after filtering
filter_df = df[df.groupby('AIRPORT')['AIRPORT'].transform('count') >= 500]
filter_df = filter_df[filter_df.groupby('SPECIES')['SPECIES'].transform('count') >= 100]

print(f'# of airports: {len(filter_df['AIRPORT'].unique())}\n')
print(filter_df['STATE'].unique(),'\n')
print(f'# of species: {len(filter_df['SPECIES'].unique())}')

# of airports: 89

<StringArray>
['CA', 'IL', 'NY', 'CT', 'FL', 'MI', 'RI', 'DC', 'NJ', 'HI', 'PA', 'VA', 'KY',
 'LA', 'MA', 'UT', 'NE', 'TN', 'TX', 'OH', 'OR', 'MD', 'OK', 'AL', 'WI', 'NC',
 'WA', 'IA', 'IN', 'MO', 'ME', 'GA', 'AR', 'MN', 'SC', 'AZ', 'CO', 'NH']
Length: 38, dtype: str 

# of species: 141


In [401]:
# filter quantities
df = df[df.groupby('AIRPORT')['AIRPORT'].transform('count') >= 500]
df = df[df.groupby('SPECIES')['SPECIES'].transform('count') >= 100]

df = df[['AIRPORT', 'SPECIES']]
df

,AIRPORT,SPECIES
8,NORMAN Y. MINETA SAN JOSE INTL ARPT,American robin
15,LOS ANGELES INTL,American kestrel
18,CHICAGO MIDWAY INTL ARPT,Gulls
21,BUFFALO-NIAGARA INTL,Rock pigeon
29,CHICAGO O'HARE INTL ARPT,Blackbirds
...,...,...
339259,SARASOTA-BRADENTON INTL ARPT,Yellow-rumped warbler
339262,HARTSFIELD - JACKSON ATLANTA INTL ARPT,Chimney swift
339267,SEATTLE-TACOMA INTL,Cedar waxwing
339268,CHICAGO O'HARE INTL ARPT,Yellow-rumped warbler


# calculate collision counts

In [402]:
counts = pd.crosstab(df['SPECIES'], df['AIRPORT']).stack('AIRPORT')
counts

SPECIES            AIRPORT                                        
American barn owl  ALBANY INTL                                        0
                   ATLANTIC CITY INTL                                 0
                   AUSTIN-BERGSTROM INTL                              8
                   BALTIMORE/WASH INTL THURGOOD MARSHAL ARPT          1
                   BILL AND  HILLARY CLINTON NATL ARPT/ADAMS FIELD    4
                                                                     ..
Zebra dove         TWEED-NEW HAVEN ARPT                               0
                   WASHINGTON DULLES INTL ARPT                        0
                   WESTCHESTER COUNTY ARPT                            0
                   WILL ROGERS WORLD ARPT                             0
                   WILLIAM P HOBBY ARPT                               0
Length: 12549, dtype: int64

### separate airport and species index

In [403]:
species = counts.index.get_level_values('SPECIES').tolist()
airports = counts.index.get_level_values('AIRPORT').tolist()

counts_df = pd.DataFrame({'Species': species, 'Airport': airports})
counts_df['Count'] = counts.values
counts_df

,Species,Airport,Count
0,American barn owl,ALBANY INTL,0
1,American barn owl,ATLANTIC CITY INTL,0
2,American barn owl,AUSTIN-BERGSTROM INTL,8
3,American barn owl,BALTIMORE/WASH INTL THURGOOD MARSHAL ARPT,1
4,American barn owl,BILL AND HILLARY CLINTON NATL ARPT/ADAMS FIELD,4
...,...,...,...
12544,Zebra dove,TWEED-NEW HAVEN ARPT,0
12545,Zebra dove,WASHINGTON DULLES INTL ARPT,0
12546,Zebra dove,WESTCHESTER COUNTY ARPT,0
12547,Zebra dove,WILL ROGERS WORLD ARPT,0


### add collision counts to new dataframe format

In [404]:
# create lists of unique species names and airport names
unique_sp = sorted(list(set(species)))
unique_air = sorted(list(set(airports)))

# create list of collision count lists
counts_list = []
for sp in unique_sp:
    sp_filter = counts_df[counts_df['Species'] == sp]
    counts_list.append(sp_filter['Count'].tolist())
    
# create new dataframe format
collisions_df = pd.DataFrame(counts_list, columns=unique_air, index=unique_sp)
collisions_df

,ALBANY INTL,ATLANTIC CITY INTL,AUSTIN-BERGSTROM INTL,BALTIMORE/WASH INTL THURGOOD MARSHAL ARPT,BILL AND HILLARY CLINTON NATL ARPT/ADAMS FIELD,BLUE GRASS ARPT,BRADLEY INTL,BUFFALO-NIAGARA INTL,BURKE LAKEFRONT,CHARLESTON AFB/INTL ARPT,...,SEATTLE-TACOMA INTL,SOUTHWEST FLORIDA INTL ARPT,TAMPA INTL,TETERBORO AIRPORT,THEODORE FRANCIS GREEN STATE,TWEED-NEW HAVEN ARPT,WASHINGTON DULLES INTL ARPT,WESTCHESTER COUNTY ARPT,WILL ROGERS WORLD ARPT,WILLIAM P HOBBY ARPT
American barn owl,0,0,8,1,4,2,0,0,0,0,...,123,7,13,19,0,0,12,0,3,14
American coot,0,0,1,0,1,0,0,0,0,0,...,8,0,2,0,0,0,0,0,1,0
American crow,8,2,2,7,2,6,2,1,0,1,...,31,1,5,11,1,4,7,7,0,2
American golden-plover,3,0,2,3,1,0,3,0,7,0,...,0,0,0,4,3,2,7,1,0,2
American goldfinch,0,1,1,4,0,2,3,1,0,0,...,4,0,1,3,1,1,1,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Woodchuck,6,0,0,29,0,2,0,3,2,0,...,0,0,0,29,0,1,1,2,0,0
Yellow-bellied sapsucker,0,1,0,0,0,0,2,0,0,0,...,0,0,0,1,0,0,1,6,0,0
Yellow-crowned night heron,0,0,0,0,0,0,0,0,0,0,...,0,2,5,2,0,0,0,0,0,10
Yellow-rumped warbler,0,3,1,2,2,0,2,0,0,1,...,11,16,4,2,1,0,1,6,1,0


### verify output

In [405]:
# compare sum of collisions from counts_df and collisions_df
counts_1 = counts_df.groupby('Species')['Count'].sum().values.tolist()
counts_2 = collisions_df.sum(axis=1).values.tolist()
counts_1 == counts_2

True

# clustering

In [421]:
scaler = RobustScaler()
scaled_data = scaler.fit_transform(collisions_df)

model = AgglomerativeClustering(n_clusters=6, metric='cosine', linkage='complete')
labels = model.fit_predict(scaled_data)
labels

array([1, 1, 5, 0, 5, 1, 1, 5, 4, 3, 5, 5, 0, 3, 1, 1, 3, 5, 3, 5, 3, 1,
       5, 5, 1, 4, 1, 5, 3, 4, 5, 5, 1, 4, 2, 4, 3, 2, 1, 2, 5, 5, 3, 1,
       5, 0, 3, 2, 2, 2, 3, 0, 4, 4, 1, 3, 2, 5, 5, 4, 4, 4, 2, 1, 5, 5,
       0, 2, 3, 1, 5, 5, 0, 3, 3, 5, 1, 3, 5, 5, 3, 2, 5, 2, 1, 4, 1, 1,
       3, 1, 0, 3, 5, 5, 5, 5, 5, 1, 0, 4, 3, 3, 0, 0, 4, 4, 1, 1, 2, 4,
       5, 3, 5, 1, 5, 2, 5, 5, 4, 5, 3, 3, 1, 5, 3, 3, 1, 0, 2, 1, 1, 5,
       2, 4, 3, 5, 5, 4, 3, 4, 1])

In [422]:
pd.DataFrame(labels).value_counts()

0
5    40
1    29
3    27
4    19
2    15
0    11
Name: count, dtype: int64